In [1]:
import pandas as pd
import numpy as np

# 1. Descargar y cargar datos (MovieLens 100k)
!wget -q http://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip

columns = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('ml-100k/u.data', sep='\t', names=columns)

# 2. Split simple (80% train, 20% test)
df = df.sample(frac=1, random_state=42) # Shuffle
train_size = int(0.8 * len(df))
train_df = df[:train_size]
test_df = df[train_size:]

# 3. Definición de Baselines
def get_recommendations_random(n=10):
    all_items = train_df['item_id'].unique()
    return np.random.choice(all_items, n, replace=False)

def get_recommendations_popular(n=10):
    popular_items = train_df.groupby('item_id')['rating'].count().sort_values(ascending=False).head(n).index.tolist()
    return popular_items

def get_recommendations_user_avg(user_id, n=10):
    # Items no vistos por el usuario
    user_seen = train_df[train_df['user_id'] == user_id]['item_id'].unique()
    all_items = train_df['item_id'].unique()
    candidates = [i for i in all_items if i not in user_seen]
    # Retornamos los top N populares de los no vistos (como proxy de promedio)
    popular_unseen = train_df[train_df['item_id'].isin(candidates)].groupby('item_id')['rating'].count().sort_values(ascending=False).head(n).index.tolist()
    return popular_unseen

# 4. Evaluación (Hit Ratio @ 10)
def evaluate_baselines(test_data, n=10):
    results = {"Random": 0, "Popular": 0}
    test_users = test_data['user_id'].unique()

    # Cacheamos recomendaciones globales para velocidad
    rand_rec = get_recommendations_random(n)
    pop_rec = get_recommendations_popular(n)

    hits_rand = 0
    hits_pop = 0

    for user in test_users:
        actual_items = set(test_data[test_data['user_id'] == user]['item_id'])

        if any(item in actual_items for item in rand_rec): hits_rand += 1
        if any(item in actual_items for item in pop_rec): hits_pop += 1

    return {
        "Random HR@10": hits_rand / len(test_users),
        "Popular HR@10": hits_pop / len(test_users)
    }

# Ejecutar y mostrar resultados
print("Calculando métricas para la Propuesta H1...")
metrics = evaluate_baselines(test_df)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# 5. Análisis Descriptivo Express
print("\n--- Estadísticas para tu tabla de la H1 ---")
print(f"Usuarios únicos: {df.user_id.nunique()}")
print(f"Items únicos: {df.item_id.nunique()}")
print(f"Total de interacciones: {len(df)}")
print(f"Densidad: {len(df) / (df.user_id.nunique() * df.item_id.nunique()):.4f}")

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.base         
  inflating: ml-100k/u2.test         
  inflating: ml-100k/u3.base         
  inflating: ml-100k/u3.test         
  inflating: ml-100k/u4.base         
  inflating: ml-100k/u4.test         
  inflating: ml-100k/u5.base         
  inflating: ml-100k/u5.test         
  inflating: ml-100k/ua.base         
  inflating: ml-100k/ua.test         
  inflating: ml-100k/ub.base         
  inflating: ml-100k/ub.test         
Calculando métricas para la Propuesta H1...
Random HR@10: 0.0160
Popular HR@10: 0.